# Steam Price Modeling

This notebook keeps the experiment path that supports the final model choice: baseline models, market-scope selection, KNN comparable-game pricing, and boosted-model improvement.


In [1]:
import ast
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 140)

In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "games_price_model.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
MODEL_DATA_PATH = DATA_DIR / "games_price_model.csv"

## 1. Load modeling data

The feature-engineering notebook already filters to valid paid games with estimated list prices between `$0.49` and `$99.99`.

In [3]:
def parse_list(value):
    if pd.isna(value):
        return []
    return ast.literal_eval(value)


df = pd.read_csv(MODEL_DATA_PATH)
for col in ["genre_list", "tag_list", "category_list"]:
    df[col] = df[col].apply(parse_list)

print(f"Shape: {df.shape}")
print("Valid price targets:", df["valid_price_target"].sum())
print("Estimated list price range:", df["estimated_list_price"].min(), df["estimated_list_price"].max())
df.head()

Shape: (96062, 42)
Valid price targets: 96062
Estimated list price range: 0.49 99.99


,AppID,Name,current_price,Discount,has_active_discount,estimated_list_price,log_list_price,price_target_source,valid_price_target,Release date,Release date parsed,Estimated owners,Genres,Tags,Categories,Developers,Publishers,Windows,Mac,Linux,required_age_clipped,platform_count,Release year,release_age_years,dlc_count_clipped,achievements_clipped,log1p_dlc_count,log1p_achievements,genre_count,tag_count,category_count,genre_list,tag_list,category_list,review_count,positive_share,log1p_review_count,log1p_recommendations,log1p_peak_ccu,log1p_avg_playtime_forever,log1p_median_playtime_forever,Metacritic score
0,496350,Supipara - Chapter 1 Spring Has Come!,5.24,65,True,14.97,2.770712,reconstructed,True,"Jul 29, 2016",2016-07-29,0 - 20000,Adventure,"Adventure,Visual Novel,Anime,Cute","Single-player,Steam Trading Cards,Steam Cloud,Family Sharing",minori,MangaGamer,True,False,False,0,1,2016,10.0,0,0,0.000000,0.000000,1,4,4,[Adventure],"[Adventure, Visual Novel, Anime, Cute]","[Single-player, Steam Trading Cards, Steam Cloud, Family Sharing]",255,0.988235,5.545177,5.446737,0.000000,2.197225,2.197225,0
1,1034400,Mystery Solitaire The Black Raven,4.99,0,False,4.99,1.790091,observed,True,"May 6, 2019",2019-05-06,0 - 20000,Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Object,2D,Colorful,Stylized,Logic,Mystery,Atmospheric,Family Friendly,PvE,Tutorial,Singleplayer...","Single-player,Family Sharing",Somer Games,8floor,True,True,False,0,2,2019,7.0,0,0,0.000000,0.000000,1,16,2,[Casual],"[Casual, Card Game, Solitaire, Puzzle, Hidden Object, 2D, Colorful, Stylized, Logic, Mystery, Atmospheric, Family Friendly, PvE, Tutoria...","[Single-player, Family Sharing]",24,0.875000,3.218876,0.000000,0.000000,0.000000,0.000000,0
2,3292190,버튜버 파라노이아 - Vtuber Paranoia,8.99,0,False,8.99,2.301585,observed,True,"Oct 31, 2024",2024-10-31,0 - 20000,"Casual,Indie,Simulation",NaN,"Single-player,Steam Achievements,Family Sharing",유진게임즈,유진게임즈,True,False,False,0,1,2024,2.0,1,19,0.693147,2.995732,3,0,3,"[Casual, Indie, Simulation]",[],"[Single-player, Steam Achievements, Family Sharing]",0,NaN,0.000000,0.000000,0.693147,0.000000,0.000000,0
3,3631080,Maze Quest VR,4.99,0,False,4.99,1.790091,observed,True,"Apr 24, 2025",2025-04-24,0 - 20000,"Action,Early Access",NaN,"Single-player,VR Only,Steam Leaderboards,Family Sharing",Reality Expanded LLC,Reality Expanded LLC,True,False,False,0,1,2025,1.0,0,0,0.000000,0.000000,2,0,4,"[Action, Early Access]",[],"[Single-player, VR Only, Steam Leaderboards, Family Sharing]",0,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0
4,1654170,Agony VR,13.99,0,False,13.99,2.707383,observed,True,"Apr 5, 2023",2023-04-05,0 - 20000,"Action,Adventure",NaN,"Single-player,Tracked Controller Support,VR Only,Family Sharing",Ignibit,"Ignibit,Madmind Studio",True,False,False,0,1,2023,3.0,0,0,0.000000,0.000000,2,0,4,"[Action, Adventure]",[],"[Single-player, Tracked Controller Support, VR Only, Family Sharing]",0,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0


In [4]:
df["estimated_list_price"].describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).to_frame("estimated_list_price")

,estimated_list_price
count,96062.000000
mean,7.957556
std,8.848327
min,0.490000
1%,0.490000
5%,0.990000
10%,0.990000
25%,2.390000
50%,4.990000
75%,9.990000


## 2. Feature sets

This first pass uses pre-release features only. Outcome variables such as reviews, recommendations, CCU, and playtime are held back for a later gamer-facing value model.

In [5]:
TARGET = "log_list_price"

ID_COLS = ["AppID", "Name"]
NUMERIC_FEATURES = [
    "required_age_clipped",
    "platform_count",
    "Release year",
    "release_age_years",
    "dlc_count_clipped",
    "achievements_clipped",
    "log1p_dlc_count",
    "log1p_achievements",
    "genre_count",
    "tag_count",
    "category_count",
]
BOOLEAN_FEATURES = ["Windows", "Mac", "Linux"]
MULTILABEL_FEATURES = ["genre_list", "tag_list", "category_list"]

TAG_MIN_COUNT = 100
CATEGORY_MIN_COUNT = 100


def label_counts(series):
    return series.explode().loc[lambda s: s.notna() & s.ne("")].value_counts()


genre_vocab = sorted(label_counts(df["genre_list"]).index.tolist())
tag_vocab = sorted(label_counts(df["tag_list"]).loc[lambda s: s >= TAG_MIN_COUNT].index.tolist())
category_vocab = sorted(label_counts(df["category_list"]).loc[lambda s: s >= CATEGORY_MIN_COUNT].index.tolist())

feature_summary = pd.Series({
    "rows": len(df),
    "numeric_features": len(NUMERIC_FEATURES),
    "boolean_features": len(BOOLEAN_FEATURES),
    "genre_features": len(genre_vocab),
    "tag_features": len(tag_vocab),
    "category_features": len(category_vocab),
})
feature_summary.to_frame("count")

,count
rows,96062
numeric_features,11
boolean_features,3
genre_features,33
tag_features,390
category_features,49


## 3. Train/test split

The split is random for the baseline. Later, we can test time-aware validation if release year proves important.

In [6]:
train_df, test_df = train_test_split(df, test_size=0.20, random_state=42)

y_train = train_df[TARGET].to_numpy()
y_test = test_df[TARGET].to_numpy()

split_summary = pd.DataFrame({
    "split": ["train", "test"],
    "rows": [len(train_df), len(test_df)],
    "median_price": [train_df["estimated_list_price"].median(), test_df["estimated_list_price"].median()],
    "mean_price": [train_df["estimated_list_price"].mean(), test_df["estimated_list_price"].mean()],
    "reconstructed_pct": [
        train_df["price_target_source"].eq("reconstructed").mean() * 100,
        test_df["price_target_source"].eq("reconstructed").mean() * 100,
    ],
})
split_summary.round(2)

,split,rows,median_price,mean_price,reconstructed_pct
0,train,76849,4.99,7.94,42.2
1,test,19213,4.99,8.02,42.5


## 4. Build sparse feature matrix

Ridge and KNN both use the same pre-release feature matrix. Numeric features are imputed and standardized; multi-label features are multi-hot encoded.

In [7]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

numeric_transformer = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("boolean", SimpleImputer(strategy="most_frequent"), BOOLEAN_FEATURES),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)

genre_binarizer = MultiLabelBinarizer(classes=genre_vocab, sparse_output=True)
tag_binarizer = MultiLabelBinarizer(classes=tag_vocab, sparse_output=True)
category_binarizer = MultiLabelBinarizer(classes=category_vocab, sparse_output=True)


def keep_vocab(items, vocab):
    vocab = set(vocab)
    return [item for item in items if item in vocab]


def build_feature_matrix(frame, fit=False):
    frame_for_transform = frame.copy()
    frame_for_transform[BOOLEAN_FEATURES] = frame_for_transform[BOOLEAN_FEATURES].astype(float)
    frame_for_transform["genre_list"] = frame_for_transform["genre_list"].apply(lambda items: keep_vocab(items, genre_vocab))
    frame_for_transform["tag_list"] = frame_for_transform["tag_list"].apply(lambda items: keep_vocab(items, tag_vocab))
    frame_for_transform["category_list"] = frame_for_transform["category_list"].apply(lambda items: keep_vocab(items, category_vocab))

    if fit:
        numeric_matrix = numeric_transformer.fit_transform(frame_for_transform)
        genre_matrix = genre_binarizer.fit_transform(frame_for_transform["genre_list"])
        tag_matrix = tag_binarizer.fit_transform(frame_for_transform["tag_list"])
        category_matrix = category_binarizer.fit_transform(frame_for_transform["category_list"])
    else:
        numeric_matrix = numeric_transformer.transform(frame_for_transform)
        genre_matrix = genre_binarizer.transform(frame_for_transform["genre_list"])
        tag_matrix = tag_binarizer.transform(frame_for_transform["tag_list"])
        category_matrix = category_binarizer.transform(frame_for_transform["category_list"])

    numeric_matrix = sparse.csr_matrix(numeric_matrix)
    return sparse.hstack([numeric_matrix, genre_matrix, tag_matrix, category_matrix], format="csr")


X_train = build_feature_matrix(train_df, fit=True)
X_test = build_feature_matrix(test_df, fit=False)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Sparsity:", 1 - X_train.nnz / (X_train.shape[0] * X_train.shape[1]))

X_train: (76849, 486)
X_test: (19213, 486)
Sparsity: 0.9365216604825014


In [8]:
feature_names = (
    NUMERIC_FEATURES
    + BOOLEAN_FEATURES
    + [f"genre={name}" for name in genre_binarizer.classes_]
    + [f"tag={name}" for name in tag_binarizer.classes_]
    + [f"category={name}" for name in category_binarizer.classes_]
)

print("Feature names:", len(feature_names))
feature_names[:20]

Feature names: 486


['required_age_clipped',
 'platform_count',
 'Release year',
 'release_age_years',
 'dlc_count_clipped',
 'achievements_clipped',
 'log1p_dlc_count',
 'log1p_achievements',
 'genre_count',
 'tag_count',
 'category_count',
 'Windows',
 'Mac',
 'Linux',
 'genre=360 Video',
 'genre=Accounting',
 'genre=Action',
 'genre=Adventure',
 'genre=Animation & Modeling',
 'genre=Audio Production']

## 5. Evaluation helpers

Models train on log price, but evaluation is reported in dollars for business readability.

In [9]:
def to_price(log_values):
    return np.expm1(log_values).clip(min=0.49)


def evaluate_predictions(model_name, y_true_log, y_pred_log):
    actual = to_price(y_true_log)
    predicted = to_price(y_pred_log)
    errors = np.abs(actual - predicted)
    rmse = mean_squared_error(actual, predicted) ** 0.5

    return {
        "model": model_name,
        "mae_dollars": mean_absolute_error(actual, predicted),
        "median_ae_dollars": median_absolute_error(actual, predicted),
        "rmse_dollars": rmse,
        "mean_actual_price": actual.mean(),
        "mean_predicted_price": predicted.mean(),
        "pct_within_25pct": (np.maximum(actual, predicted) / np.minimum(actual, predicted) <= 1.25).mean() * 100,
        "pct_within_50pct": (np.maximum(actual, predicted) / np.minimum(actual, predicted) <= 1.50).mean() * 100,
    }


def price_band_table(frame, y_pred_log):
    result = frame[["estimated_list_price"]].copy()
    result["predicted_price"] = to_price(y_pred_log)
    result["abs_error"] = (result["estimated_list_price"] - result["predicted_price"]).abs()
    result["price_band"] = pd.cut(
        result["estimated_list_price"],
        bins=[0, 2.99, 4.99, 9.99, 19.99, 49.99, 100],
        labels=["0.49-2.99", "3-4.99", "5-9.99", "10-19.99", "20-49.99", "50-99.99"],
        include_lowest=True,
    )
    return (
        result.groupby("price_band", observed=False)
        .agg(
            games=("estimated_list_price", "size"),
            median_actual=("estimated_list_price", "median"),
            median_predicted=("predicted_price", "median"),
            mae=("abs_error", "mean"),
            median_ae=("abs_error", "median"),
        )
    )

## 6. Median baseline

This baseline predicts the same median log price for every game. Every real model should beat it.

In [10]:
median_model = DummyRegressor(strategy="median")
median_model.fit(X_train, y_train)
median_pred_log = median_model.predict(X_test)

results = [evaluate_predictions("Median baseline", y_test, median_pred_log)]
pd.DataFrame(results).round(3)

,model,mae_dollars,median_ae_dollars,rmse_dollars,mean_actual_price,mean_predicted_price,pct_within_25pct,pct_within_50pct
0,Median baseline,5.393,3.03,9.382,8.017,4.99,17.608,28.309


## 7. Ridge regression baseline

Ridge is a strong first model for sparse multi-hot metadata. It is fast, stable, and interpretable enough for baseline analysis.

In [11]:
ridge_model = Ridge(alpha=10.0, random_state=42)
ridge_model.fit(X_train, y_train)
ridge_pred_log = ridge_model.predict(X_test)

results.append(evaluate_predictions("Ridge", y_test, ridge_pred_log))
pd.DataFrame(results).round(3)

,model,mae_dollars,median_ae_dollars,rmse_dollars,mean_actual_price,mean_predicted_price,pct_within_25pct,pct_within_50pct
0,Median baseline,5.393,3.030,9.382,8.017,4.990,17.608,28.309
1,Ridge,4.510,2.707,7.688,8.017,6.182,20.189,36.408


In [12]:
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef_log_price": ridge_model.coef_,
})

display(coef_df.sort_values("coef_log_price", ascending=False).head(20))
display(coef_df.sort_values("coef_log_price").head(20))

,feature,coef_log_price
44,genre=Video Production,0.332364
27,genre=Game Development,0.312831
483,category=VR Only,0.254412
9,tag_count,0.250576
287,tag=Otome,0.246554
23,genre=Early Access,0.245162
485,category=VR Supported,0.235017
358,tag=Software Training,0.199207
403,tag=Trains,0.199018
447,category=Full controller support,0.194383


,feature,coef_log_price
446,category=Family Sharing,-0.554068
43,genre=Utilities,-0.397698
319,tag=RPGMaker,-0.247686
169,tag=Experience,-0.243095
350,tag=Short,-0.234887
117,tag=Clicker,-0.232201
257,tag=Minimalist,-0.218116
33,genre=Photo Editing,-0.214228
473,category=Steam Achievements,-0.210877
252,tag=Memes,-0.206392


## 8. KNN median-price baseline

This baseline predicts each test game's price as the median list price of its nearest training neighbors. It is slower than Ridge, but useful because it resembles the future comparable-games layer.

In [13]:
K_NEIGHBORS = 25

knn_model = NearestNeighbors(n_neighbors=K_NEIGHBORS, metric="cosine", algorithm="brute", n_jobs=-1)
knn_model.fit(X_train)

neighbor_distances, neighbor_indices = knn_model.kneighbors(X_test)
neighbor_prices = train_df.iloc[neighbor_indices.ravel()]["estimated_list_price"].to_numpy().reshape(neighbor_indices.shape)
knn_pred_price = np.median(neighbor_prices, axis=1)
knn_pred_log = np.log1p(knn_pred_price)

results.append(evaluate_predictions(f"KNN median k={K_NEIGHBORS}", y_test, knn_pred_log))
pd.DataFrame(results).round(3)

,model,mae_dollars,median_ae_dollars,rmse_dollars,mean_actual_price,mean_predicted_price,pct_within_25pct,pct_within_50pct
0,Median baseline,5.393,3.030,9.382,8.017,4.990,17.608,28.309
1,Ridge,4.510,2.707,7.688,8.017,6.182,20.189,36.408
2,KNN median k=25,4.311,2.750,7.406,8.017,6.354,22.979,37.292


## 9. Model comparison and error by price band

In [14]:
comparison = pd.DataFrame(results).sort_values("mae_dollars")
comparison.round(3)

,model,mae_dollars,median_ae_dollars,rmse_dollars,mean_actual_price,mean_predicted_price,pct_within_25pct,pct_within_50pct
2,KNN median k=25,4.311,2.750,7.406,8.017,6.354,22.979,37.292
1,Ridge,4.510,2.707,7.688,8.017,6.182,20.189,36.408
0,Median baseline,5.393,3.030,9.382,8.017,4.990,17.608,28.309


In [15]:
display(price_band_table(test_df, median_pred_log).round(2).rename_axis("Median baseline"))
display(price_band_table(test_df, ridge_pred_log).round(2).rename_axis("Ridge"))
display(price_band_table(test_df, knn_pred_log).round(2).rename_axis("KNN median"))

,games,median_actual,median_predicted,mae,median_ae
Median baseline,,,,,
0.49-2.99,6567,1.97,4.99,3.22,3.02
3-4.99,3834,4.96,4.99,0.41,0.03
5-9.99,4641,7.99,4.99,3.29,3.00
10-19.99,3123,14.99,4.99,10.71,10.00
20-49.99,921,29.98,4.99,26.92,24.99
50-99.99,127,59.98,4.99,58.07,54.99


,games,median_actual,median_predicted,mae,median_ae
Ridge,,,,,
0.49-2.99,6567,1.97,3.95,2.66,2.33
3-4.99,3834,4.96,4.62,1.71,1.22
5-9.99,4641,7.99,5.44,3.24,2.95
10-19.99,3123,14.99,7.41,7.71,7.54
20-49.99,921,29.98,11.13,19.18,18.61
50-99.99,127,59.98,12.32,46.12,48.92


,games,median_actual,median_predicted,mae,median_ae
KNN median,,,,,
0.49-2.99,6567,1.97,3.98,2.50,2.00
3-4.99,3834,4.96,4.95,1.88,1.01
5-9.99,4641,7.99,4.99,3.32,3.00
10-19.99,3123,14.99,7.99,7.32,7.00
20-49.99,921,29.98,12.99,17.37,17.50
50-99.99,127,59.98,20.99,38.83,40.01


## 10. Initial modeling takeaways

The full valid-price model is useful as a first baseline: both Ridge and KNN beat the median-price baseline. Inspection also shows a problem: the broad Steam dataset includes many zero-traction or very low-traction games, which may dilute the market signal for games that players and developers actually compare against.

To test stricter market scopes fairly, each scope needs its own local median baseline. Filtered scopes have different price distributions, so comparing a filtered model directly against the full-valid baseline would be misleading.



## 11. Market-scope sensitivity

These scopes test whether removing near-zero-traction games improves price modeling. The filters are based on observable market evidence, not subjective title quality.



In [16]:
scope_masks = {
    "full_valid": pd.Series(True, index=df.index),
    "review_count >= 10": df["review_count"] >= 10,
    "review_count >= 50": df["review_count"] >= 50,
    "recommendations > 0": df["log1p_recommendations"] > 0,
}

scope_rows = []
for scope_name, mask in scope_masks.items():
    scope_df = df.loc[mask].copy()
    scope_rows.append({
        "scope": scope_name,
        "rows": len(scope_df),
        "pct_of_full": len(scope_df) / len(df) * 100,
        "median_price": scope_df["estimated_list_price"].median(),
        "mean_price": scope_df["estimated_list_price"].mean(),
        "median_review_count": scope_df["review_count"].median(),
        "median_recommendations_log": scope_df["log1p_recommendations"].median(),
        "reconstructed_pct": scope_df["price_target_source"].eq("reconstructed").mean() * 100,
    })

scope_summary_df = pd.DataFrame(scope_rows)
scope_summary_df.round(2)


,scope,rows,pct_of_full,median_price,mean_price,median_review_count,median_recommendations_log,reconstructed_pct
0,full_valid,96062,100.00,4.99,7.96,10.0,0.00,42.26
1,review_count >= 10,49139,51.15,5.99,9.48,60.0,0.00,53.65
2,review_count >= 50,26542,27.63,9.90,11.83,239.0,5.29,63.46
3,recommendations > 0,20211,21.04,9.99,13.55,403.0,6.02,68.24


In [17]:
def train_models_for_scope(scope_df, alpha=10.0, random_state=42):
    train_scope_df, test_scope_df = train_test_split(scope_df, test_size=0.20, random_state=random_state)
    y_scope_train = train_scope_df[TARGET].to_numpy()
    y_scope_test = test_scope_df[TARGET].to_numpy()

    X_scope_train = build_feature_matrix(train_scope_df, fit=True)
    X_scope_test = build_feature_matrix(test_scope_df, fit=False)

    median_model = DummyRegressor(strategy="median")
    median_model.fit(X_scope_train, y_scope_train)
    median_pred_log = median_model.predict(X_scope_test)

    ridge_model = Ridge(alpha=alpha, random_state=random_state)
    ridge_model.fit(X_scope_train, y_scope_train)
    ridge_pred_log = ridge_model.predict(X_scope_test)

    median_metrics = evaluate_predictions("Median baseline", y_scope_test, median_pred_log)
    ridge_metrics = evaluate_predictions("Ridge", y_scope_test, ridge_pred_log)

    shared_metrics = {
        "train_rows": len(train_scope_df),
        "test_rows": len(test_scope_df),
        "test_median_price": test_scope_df["estimated_list_price"].median(),
        "test_mean_price": test_scope_df["estimated_list_price"].mean(),
    }
    median_metrics.update(shared_metrics)
    ridge_metrics.update(shared_metrics)

    return {
        "median_metrics": median_metrics,
        "ridge_metrics": ridge_metrics,
        "median_model": median_model,
        "ridge_model": ridge_model,
        "train_df": train_scope_df,
        "test_df": test_scope_df,
        "X_train": X_scope_train,
        "X_test": X_scope_test,
        "median_pred_log": median_pred_log,
        "ridge_pred_log": ridge_pred_log,
    }


In [18]:
scope_models = {}
scope_metrics = []

for scope_name, mask in scope_masks.items():
    scope_df = df.loc[mask].copy()
    scope_result = train_models_for_scope(scope_df)
    scope_models[scope_name] = scope_result

    for metrics in [scope_result["median_metrics"], scope_result["ridge_metrics"]]:
        row = metrics.copy()
        row["scope"] = scope_name
        scope_metrics.append(row)

scope_results = pd.DataFrame(scope_metrics)
scope_results = scope_results[[
    "scope", "model", "train_rows", "test_rows", "mae_dollars", "median_ae_dollars", "rmse_dollars",
    "pct_within_25pct", "pct_within_50pct", "test_median_price", "test_mean_price"
]]

baseline_mae = (
    scope_results.loc[scope_results["model"].eq("Median baseline"), ["scope", "mae_dollars"]]
    .rename(columns={"mae_dollars": "scope_median_baseline_mae"})
)
scope_results = scope_results.merge(baseline_mae, on="scope", how="left")
scope_results["mae_improvement_vs_scope_median"] = (
    scope_results["scope_median_baseline_mae"] - scope_results["mae_dollars"]
)
scope_results["mae_improvement_pct_vs_scope_median"] = (
    scope_results["mae_improvement_vs_scope_median"] / scope_results["scope_median_baseline_mae"] * 100
)

scope_results.sort_values(["scope", "model"]).round(3)


,scope,model,train_rows,test_rows,mae_dollars,median_ae_dollars,rmse_dollars,pct_within_25pct,pct_within_50pct,test_median_price,test_mean_price,scope_median_baseline_mae,mae_improvement_vs_scope_median,mae_improvement_pct_vs_scope_median
0,full_valid,Median baseline,76849,19213,5.393,3.030,9.382,17.608,28.309,4.99,8.017,5.393,0.000,0.000
1,full_valid,Ridge,76849,19213,4.510,2.707,7.688,20.189,36.408,4.99,8.017,5.393,0.883,16.375
6,recommendations > 0,Median baseline,16168,4043,8.215,5.990,12.149,17.339,33.119,9.98,13.392,8.215,0.000,0.000
7,recommendations > 0,Ridge,16168,4043,6.160,4.078,9.090,26.193,44.200,9.98,13.392,8.215,2.056,25.020
2,review_count >= 10,Median baseline,39311,9828,6.329,4.000,10.245,20.045,25.743,5.99,9.405,6.329,0.000,0.000
3,review_count >= 10,Ridge,39311,9828,4.911,3.104,7.879,21.530,38.472,5.99,9.405,6.329,1.417,22.394
4,review_count >= 50,Median baseline,21233,5309,7.603,5.910,11.133,19.891,26.879,9.90,11.724,7.603,0.000,0.000
5,review_count >= 50,Ridge,21233,5309,5.693,3.743,8.639,24.053,41.571,9.90,11.724,7.603,1.910,25.123


## 12. KNN on selected market scope

If a filtered scope improves Ridge performance, run the KNN median-price baseline on that scope as a comparable-games-style estimator. The default selected scope is `review_count >= 10`, which removes completely unvalidated games while keeping enough market breadth.



In [19]:
SELECTED_SCOPE = "review_count >= 10"

selected = scope_models[SELECTED_SCOPE]
selected_train_df = selected["train_df"]
selected_test_df = selected["test_df"]
X_selected_train = selected["X_train"]
X_selected_test = selected["X_test"]
y_selected_test = selected_test_df[TARGET].to_numpy()

selected_knn = NearestNeighbors(n_neighbors=K_NEIGHBORS, metric="cosine", algorithm="brute", n_jobs=-1)
selected_knn.fit(X_selected_train)
selected_distances, selected_indices = selected_knn.kneighbors(X_selected_test)
selected_neighbor_prices = selected_train_df.iloc[selected_indices.ravel()]["estimated_list_price"].to_numpy().reshape(selected_indices.shape)
selected_knn_pred_price = np.median(selected_neighbor_prices, axis=1)
selected_knn_pred_log = np.log1p(selected_knn_pred_price)

selected_results = [
    evaluate_predictions(f"Median baseline ({SELECTED_SCOPE})", y_selected_test, selected["median_pred_log"]),
    evaluate_predictions(f"Ridge ({SELECTED_SCOPE})", y_selected_test, selected["ridge_pred_log"]),
    evaluate_predictions(f"KNN median k={K_NEIGHBORS} ({SELECTED_SCOPE})", y_selected_test, selected_knn_pred_log),
]
selected_results_df = pd.DataFrame(selected_results)
selected_baseline_mae = selected_results_df.loc[
    selected_results_df["model"].str.startswith("Median baseline"), "mae_dollars"
].iloc[0]
selected_results_df["mae_improvement_vs_scope_median"] = selected_baseline_mae - selected_results_df["mae_dollars"]
selected_results_df["mae_improvement_pct_vs_scope_median"] = (
    selected_results_df["mae_improvement_vs_scope_median"] / selected_baseline_mae * 100
)
selected_results_df.round(3)


,model,mae_dollars,median_ae_dollars,rmse_dollars,mean_actual_price,mean_predicted_price,pct_within_25pct,pct_within_50pct,mae_improvement_vs_scope_median,mae_improvement_pct_vs_scope_median
0,Median baseline (review_count >= 10),6.329,4.000,10.245,9.405,5.990,20.045,25.743,0.000,0.000
1,Ridge (review_count >= 10),4.911,3.104,7.879,9.405,7.594,21.530,38.472,1.417,22.394
2,KNN median k=25 (review_count >= 10),4.813,3.000,7.742,9.405,7.977,25.275,39.662,1.515,23.943


In [20]:
selected_predictions = selected_test_df[[
    "AppID", "Name", "estimated_list_price", "current_price", "Discount", "price_target_source",
    "Genres", "Tags", "Categories", "review_count", "log1p_recommendations"
]].copy()
selected_predictions["predicted_market_price"] = to_price(selected_knn_pred_log)
selected_predictions["price_ratio"] = selected_predictions["estimated_list_price"] / selected_predictions["predicted_market_price"]
selected_predictions["price_alignment"] = pd.cut(
    selected_predictions["price_ratio"],
    bins=[0, 0.60, 1.50, np.inf],
    labels=["below-market", "market-aligned", "above-market"],
)

POPULAR_EXAMPLE_COLS = [
    "Name", "estimated_list_price", "predicted_market_price", "price_ratio", "price_alignment",
    "Discount", "price_target_source", "review_count", "log1p_recommendations",
]

selected_predictions.sort_values(
    ["log1p_recommendations", "review_count"], ascending=False
).loc[:, POPULAR_EXAMPLE_COLS].head(20)



,Name,estimated_list_price,predicted_market_price,price_ratio,price_alignment,Discount,price_target_source,review_count,log1p_recommendations
51155,HELLDIVERS™ 2,39.99,49.97,0.800280,market-aligned,20,reconstructed,1017635,13.577903
36828,Dead by Daylight,19.97,29.96,0.666555,market-aligned,60,reconstructed,797216,13.352402
11198,ARK: Survival Evolved,14.98,19.99,0.749375,market-aligned,34,reconstructed,730170,13.248252
88551,The Forest,19.96,14.96,1.334225,market-aligned,77,reconstructed,627791,13.228018
30254,Fallout 4,19.97,29.93,0.667224,market-aligned,60,reconstructed,392287,12.529941
55137,Hades,24.97,15.97,1.563557,above-market,70,reconstructed,279741,12.513469
59090,Mount & Blade II: Bannerlord,49.98,29.93,1.669896,above-market,50,reconstructed,264391,12.287537
31110,Borderlands 2,19.96,19.96,1.000000,market-aligned,75,reconstructed,295218,12.272366
51110,Hunt: Showdown 1896,29.98,39.98,0.749875,market-aligned,55,reconstructed,248220,12.222276
78644,ULTRAKILL,24.99,8.98,2.782851,above-market,30,reconstructed,169751,12.064284


## 13. Updated modeling decision

Current decisions after baseline inspection and scope sensitivity:

- Use `review_count >= 10` as the main practical market scope. It removes zero-traction games while keeping about half the valid paid dataset.
- Keep `full_valid` as a broad Steam-market reference scope.
- Compare every filtered scope against its own local median baseline, not against the full-valid baseline.
- Keep Ridge as the explainable pricing baseline.
- Treat KNN as both a competitive baseline and the natural bridge to comparable-game recommendations.
- Tune KNN `k` before treating `k=25` as a final choice.
- Keep fixed ratio labels as the main business-facing price-alignment rule, and inspect threshold sensitivity before locking the final cutoffs.
- Defer boosting until after the alignment rule is settled. If external dependencies are allowed, XGBoost is likely a better next boosting candidate than forcing sklearn boosting onto sparse metadata.



## 14. KNN `k` sensitivity on selected scope

The previous KNN result used `k=25` as a reasonable starting point. This section checks whether that choice is stable on the selected `review_count >= 10` scope.



In [21]:
def evaluate_knn_for_k(k, X_train, X_test, train_frame, y_test_log):
    model = NearestNeighbors(n_neighbors=k, metric="cosine", algorithm="brute", n_jobs=-1)
    model.fit(X_train)
    distances, indices = model.kneighbors(X_test)
    neighbor_prices = train_frame.iloc[indices.ravel()]["estimated_list_price"].to_numpy().reshape(indices.shape)
    pred_price = np.median(neighbor_prices, axis=1)
    pred_log = np.log1p(pred_price)
    metrics = evaluate_predictions(f"KNN median k={k}", y_test_log, pred_log)
    metrics["k"] = k
    return metrics, pred_log

K_VALUES = [15]
# Uncomment the line below to reexplore, results point to k=15 as a strong choice for this dataset
# K_VALUES = [3, 5, 10, 15, 25, 50, 75, 100]

knn_k_metrics = []
knn_k_predictions = {}

for k in K_VALUES:
    metrics, pred_log = evaluate_knn_for_k(
        k, X_selected_train, X_selected_test, selected_train_df, y_selected_test
    )
    knn_k_metrics.append(metrics)
    knn_k_predictions[k] = pred_log


In [22]:
knn_k_results = pd.DataFrame(knn_k_metrics)
selected_baseline_mae = selected_results_df.loc[
    selected_results_df["model"].str.startswith("Median baseline"), "mae_dollars"
].iloc[0]
knn_k_results["mae_improvement_vs_scope_median"] = selected_baseline_mae - knn_k_results["mae_dollars"]
knn_k_results["mae_improvement_pct_vs_scope_median"] = (
    knn_k_results["mae_improvement_vs_scope_median"] / selected_baseline_mae * 100
)
knn_k_results = knn_k_results[[
    "k", "mae_dollars", "median_ae_dollars", "rmse_dollars",
    "pct_within_25pct", "pct_within_50pct",
    "mae_improvement_vs_scope_median", "mae_improvement_pct_vs_scope_median"
]].sort_values("mae_dollars")

BEST_K = int(knn_k_results.iloc[0]["k"])
best_knn_pred_log = knn_k_predictions[BEST_K]

print("Best k by MAE:", BEST_K)
knn_k_results.round(3)


Best k by MAE: 15


,k,mae_dollars,median_ae_dollars,rmse_dollars,pct_within_25pct,pct_within_50pct,mae_improvement_vs_scope_median,mae_improvement_pct_vs_scope_median
0,15,4.811,3.0,7.74,25.702,40.191,1.517,23.978


## 15. Boosting benchmark on compressed sparse features

For a sparse multi-hot matrix like ours, the workable boosting route in sklearn is to compress the features with TruncatedSVD first, then fit a histogram-based gradient boosting regressor on the dense latent representation.

This is a benchmark, not the main interpretable model. If it clearly beats Ridge on the selected scope, we can consider it as a stronger final benchmark; otherwise Ridge stays the cleaner choice.



In [23]:
BOOSTING_COMPONENTS = [32, 64, 96, 128]
BOOSTING_RANDOM_STATE = 42


def evaluate_boosting_with_svd(n_components, X_train, X_test, y_train_log, y_test_log):
    svd = TruncatedSVD(n_components=n_components, random_state=BOOSTING_RANDOM_STATE)
    X_train_dense = svd.fit_transform(X_train)
    X_test_dense = svd.transform(X_test)

    model = HistGradientBoostingRegressor(
        loss="squared_error",
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=31,
        min_samples_leaf=20,
        l2_regularization=0.0,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=BOOSTING_RANDOM_STATE,
    )
    model.fit(X_train_dense, y_train_log)
    pred_log = model.predict(X_test_dense)
    metrics = evaluate_predictions(f"HistGB + SVD ({n_components} comps)", y_test_log, pred_log)
    metrics["n_components"] = n_components
    metrics["svd_explained_variance"] = svd.explained_variance_ratio_.sum()
    return metrics, pred_log


boosting_metrics = []
boosting_predictions = {}
y_selected_train = selected_train_df[TARGET].to_numpy()

for n_components in BOOSTING_COMPONENTS:
    metrics, pred_log = evaluate_boosting_with_svd(
        n_components,
        X_selected_train,
        X_selected_test,
        y_selected_train,
        y_selected_test,
    )
    boosting_metrics.append(metrics)
    boosting_predictions[n_components] = pred_log

boosting_results_df = pd.DataFrame(boosting_metrics)
selected_scope_baseline_mae = selected_results_df.loc[
    selected_results_df["model"].eq("Median baseline (review_count >= 10)"), "mae_dollars"
].iloc[0]
boosting_results_df["mae_improvement_vs_scope_median"] = (
    selected_scope_baseline_mae - boosting_results_df["mae_dollars"]
)
boosting_results_df["mae_improvement_pct_vs_scope_median"] = (
    boosting_results_df["mae_improvement_vs_scope_median"] / selected_scope_baseline_mae * 100
)
boosting_results_df = boosting_results_df[[
    "n_components", "mae_dollars", "median_ae_dollars", "rmse_dollars",
    "pct_within_25pct", "pct_within_50pct", "svd_explained_variance",
    "mae_improvement_vs_scope_median", "mae_improvement_pct_vs_scope_median",
]].sort_values("mae_dollars")

BEST_BOOSTING_COMPONENTS = int(boosting_results_df.iloc[0]["n_components"])
best_boosting_pred_log = boosting_predictions[BEST_BOOSTING_COMPONENTS]

print("Best SVD components:", BEST_BOOSTING_COMPONENTS)
boosting_results_df.round(3)


Best SVD components: 96


,n_components,mae_dollars,median_ae_dollars,rmse_dollars,pct_within_25pct,pct_within_50pct,svd_explained_variance,mae_improvement_vs_scope_median,mae_improvement_pct_vs_scope_median
2,96,4.830,3.004,7.725,22.120,39.316,0.839,1.499,23.683
3,128,4.840,3.056,7.756,22.161,39.489,0.878,1.489,23.528
1,64,4.843,3.045,7.768,22.212,39.632,0.781,1.486,23.477
0,32,4.907,3.089,7.819,21.306,39.153,0.680,1.421,22.456


## 16. XGBoost benchmark on selected scope

This is the direct boosting test on the same `review_count >= 10` split. If it produces a clear win over Ridge and KNN, it can become the strongest benchmark. Otherwise, the simpler models stay in front.



In [24]:
try:
    import xgboost as xgb
except ImportError as exc:
    raise ImportError(
        "xgboost is not installed in the active environment. Install it from requirements.txt and rerun this cell."
    ) from exc

xgb_model = xgb.XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)
xgb_model.fit(X_selected_train, y_selected_train)
xgb_pred_log = xgb_model.predict(X_selected_test)

xgb_metrics = evaluate_predictions("XGBoost", y_selected_test, xgb_pred_log)
xgb_baseline_mae = selected_results_df.loc[
    selected_results_df["model"].eq("Median baseline (review_count >= 10)"), "mae_dollars"
].iloc[0]
xgb_metrics["mae_improvement_vs_scope_median"] = xgb_baseline_mae - xgb_metrics["mae_dollars"]
xgb_metrics["mae_improvement_pct_vs_scope_median"] = (
    xgb_metrics["mae_improvement_vs_scope_median"] / xgb_baseline_mae * 100
)

xgb_results_df = pd.DataFrame([xgb_metrics])[[
    "model", "mae_dollars", "median_ae_dollars", "rmse_dollars",
    "pct_within_25pct", "pct_within_50pct",
    "mae_improvement_vs_scope_median", "mae_improvement_pct_vs_scope_median",
]]

xgb_results_df.round(3)


,model,mae_dollars,median_ae_dollars,rmse_dollars,pct_within_25pct,pct_within_50pct,mae_improvement_vs_scope_median,mae_improvement_pct_vs_scope_median
0,XGBoost,4.587,2.882,7.392,23.362,41.229,1.741,27.518


## Modeling Summary

The selected developer-facing setup uses `review_count >= 10` as the practical market scope. This removes zero-traction catalog noise while preserving broad market coverage.

The modeling path is deliberately incremental:

- Median baseline: minimum benchmark for the selected price distribution.
- Ridge: fast linear baseline for sparse multi-hot Steam metadata.
- KNN median price: comparable-game pricing baseline.
- XGBoost: nonlinear improvement benchmark on the same selected split.

Dollar MAE is the primary model-selection metric because the business question is how many dollars the estimated market price misses by. Median absolute error, RMSE, and within-25% / within-50% rates are kept as supporting diagnostics. Ridge and KNN remain explainable baselines, while XGBoost is the strongest pre-release predictive benchmark in the current experiments.
